In [289]:
import pandas as pd

from statsbombpy import sb

from mplsoccer import Pitch
import matplotsoccer
import tqdm

from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl

# 1. 데이터 탐색

### i) 360데이터를 활용할 수 있는지 여부 -> match_available_360확인

In [290]:
sb.competitions()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,16,4,Europe,Champions League,male,False,False,2018/2019,2023-03-07T12:20:48.118250,2021-06-13T16:17:31.694,None,2023-03-07T12:20:48.118250
1,16,1,Europe,Champions League,male,False,False,2017/2018,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2021-01-23T21:55:30.425330
2,16,2,Europe,Champions League,male,False,False,2016/2017,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2020-07-29T05:00
3,16,27,Europe,Champions League,male,False,False,2015/2016,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2020-07-29T05:00
4,16,26,Europe,Champions League,male,False,False,2014/2015,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2020-07-29T05:00
5,16,25,Europe,Champions League,male,False,False,2013/2014,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2020-07-29T05:00
6,16,24,Europe,Champions League,male,False,False,2012/2013,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2021-07-10T13:41:45.751
7,16,23,Europe,Champions League,male,False,False,2011/2012,2021-08-27T11:26:39.802832,2021-06-13T16:17:31.694,None,2020-07-29T05:00
8,16,22,Europe,Champions League,male,False,False,2010/2011,2022-01-26T21:07:11.033473,2021-06-13T16:17:31.694,None,2022-01-26T21:07:11.033473
9,16,21,Europe,Champions League,male,False,False,2009/2010,2022-11-15T17:26:10.871011,2021-06-13T16:17:31.694,None,2022-11-15T17:26:10.871011


### ii) StatsBomb360데이터를 갖고있는 Match ID 불러오기

In [291]:
import glob
import json
import os
import tqdm
import re

File_List = glob.glob('three-sixty/*.json')

all_df = pd.DataFrame()
MATCH_ID_list = []

for i in tqdm.tqdm(File_List):
    f = open(i,'r')
    
    #event_stream데이터도 매치시키기 위해 file_name가져오기
    MATCH_ID = re.sub(r"[^0-9]", "", f.name)
    MATCH_ID_list.append(MATCH_ID)

100%|██████████| 146/146 [00:00<00:00, 33212.30it/s]


### iii) 각 Match ID가 어느 대회에 해당되는지

- sb.matches(competition_id=43,season_id=106) => 64개 // 2022년 남자 월드컵
- sb.matches(competition_id=11,season_id=90) => 0개 // 20/21 라리가
- sb.matches(competition_id=55,season_id=43) => 51개 // 2020년 남자 유로파
- sb.matches(competition_id=53,season_id=106) => 31개 2022년 여자 유로파

In [292]:
competion_list = []
for c_id, s_id in tqdm.tqdm(zip(sb.competitions().competition_id,sb.competitions().season_id)):
    try:
        df = sb.matches(competition_id=c_id,season_id=s_id)
        for id in df.match_id:
            if str(id) in MATCH_ID_list:
                competion_list.append(competion_list.append([c_id, s_id]))
    except:
        #해당 대회의 경기데이터는 존재하지 않음
        print("못 불러오는 데이터 프레임 : ",(c_id,s_id))
        
print(competion_list.count([43, 106]))
print(competion_list.count([11, 90]))
print(competion_list.count([55, 43]))
print(competion_list.count([53, 106]))

0it [00:00, ?it/s]

15it [00:05,  2.82it/s]

못 불러오는 데이터 프레임 :  (16, 76)


43it [00:16,  2.69it/s]

64
0
51
31


# 2. 총 3종류의 데이터 불러오기

### 3종류의 데이터를 event_id를 기준으로 merge할 예정

- VAEP연구에서 사용했던 event데이터 : result_name를 얻기 위함
- Events , sb.events(match_id=각 경기 id) : 패스 끝 좌표를 얻기 위함
- 360 Frames , pd.read_json(f"three-sixty/{match_id}.json") : 선수들의 위치 정보를 얻기 위함

### i) VAEP연구에서 사용한 데이터 불러오기

-sb.events에서는 pass_result를 얻을 수 없으므로 해당 데이터에서 불러오기

In [293]:
SBL = StatsBombLoader(getter="remote", creds={"user": None, "passwd": None})

#StatsBomb360데이터에서는 3종류의 데이터만 지원함(2022월드컵, 20남자 유에파, 22여자 유에파)
competition = sb.competitions()
competition = competition[(competition.competition_id == 43) & (competition.season_id == 106) |
            (competition.competition_id == 55) & (competition.season_id == 43) |
            (competition.competition_id == 53) & (competition.season_id == 106)]
competition

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
18,43,106,International,FIFA World Cup,male,False,True,2022,2023-06-24T17:17:27.911026,2023-06-24T17:18:55.629111,2023-06-24T17:18:55.629111,2023-06-24T17:17:27.911026
40,55,43,Europe,UEFA Euro,male,False,True,2020,2023-02-24T21:26:47.128979,2023-04-27T22:38:34.970148,2023-04-27T22:38:34.970148,2023-02-24T21:26:47.128979
41,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2023-02-22T23:06:39.942980,2023-06-18T16:10:51.892204,2023-06-18T16:10:51.892204,2023-02-22T23:06:39.942980


In [294]:
games = pd.concat([
    SBL.games(row.competition_id, row.season_id)
    for row in competition.itertuples()
])

games = games.reset_index().drop(columns='index')
games

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,3857256,106,43,Group Stage,3,2022-12-02 21:00:00,786,773,2,3,Stadium 974,Fernando Andrés Rapallini
1,3869151,106,43,Round of 16,4,2022-12-03 21:00:00,779,792,2,1,Ahmad bin Ali Stadium,Szymon Marciniak
2,3857257,106,43,Group Stage,3,2022-11-30 17:00:00,792,776,1,0,Al Janoub Stadium,Mustapha Ghorbal
3,3857258,106,43,Group Stage,1,2022-11-24 21:00:00,781,786,2,0,Lusail Stadium,Alireza Faghani
4,3857288,106,43,Group Stage,2,2022-11-26 12:00:00,777,792,0,1,Al Janoub Stadium,Daniel Siebert
...,...,...,...,...,...,...,...,...,...,...,...,...
141,3835334,106,53,Group Stage,2,2022-07-14 18:00:00,855,862,1,1,Academy Stadium,Lina Lehtovaara
142,3835328,106,53,Group Stage,2,2022-07-11 18:00:00,859,2457,2,0,St. Mary''s Stadium,Emikar Caldera Barrera
143,3835333,106,53,Group Stage,2,2022-07-14 21:00:00,861,854,2,1,AESSEAL New York Stadium,Cheryl Foster
144,3835321,106,53,Group Stage,1,2022-07-08 18:00:00,863,920,4,1,Stadium MK,Kateryna Monzul


In [295]:
games_verbose = tqdm.tqdm(list(games.itertuples()), desc="Loading game data")
teams, players = [], []
actions = {}

for game in games_verbose:
    # load data
    teams.append(SBL.teams(game.game_id))
    players.append(SBL.players(game.game_id))
    events = SBL.events(game.game_id)
    # convert data
    actions[game.game_id] = spadl.statsbomb.convert_to_actions(events, game.home_team_id)

teams = pd.concat(teams).drop_duplicates(subset="team_id")
players = pd.concat(players)

Loading game data: 100%|██████████| 146/146 [02:09<00:00,  1.13it/s]


In [296]:
datafolder = "./data-fifa"

# Create data folder if it doesn't exist
if not os.path.exists(datafolder):
    os.mkdir(datafolder)
    print(f"Directory {datafolder} created.")

spadl_h5 = os.path.join(datafolder, "spadl-statsbomb.h5")

# Store all spadl data in h5-file
with pd.HDFStore(spadl_h5) as spadlstore:
    spadlstore["competitions"] = competition
    spadlstore["games"] = games
    spadlstore["teams"] = teams
    spadlstore["players"] = players[['player_id', 'player_name', 'nickname']].drop_duplicates(subset='player_id')
    spadlstore["player_games"] = players[['player_id', 'game_id', 'team_id', 'is_starter', 'starting_position_id', 'starting_position_name', 'minutes_played']]
    for game_id in actions.keys():
        spadlstore[f"actions/game_{game_id}"] = actions[game_id]

/home/toc3/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3326: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['competition_stage', 'venue', 'referee'], dtype='object')]

  exec(code_obj, self.user_global_ns, self.user_ns)
/home/toc3/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3326: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['player_name', 'nickname'], dtype='object')]

  exec(code_obj, self.user_global_ns, self.user_ns)
/home/toc3/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3326: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block2_val

In [297]:
VAEP_df = pd.DataFrame()
with pd.HDFStore(spadl_h5) as spadlstore:
    games = (
        spadlstore["games"]
        .merge(spadlstore["competitions"], how='left')
        .merge(spadlstore["teams"].add_prefix('home_'), how='left')
        .merge(spadlstore["teams"].add_prefix('away_'), how='left'))
    #Select korea vs germany game at World Cup
    game = games
    
    for i in range(len(game.game_id)):
        game_id = game.game_id.values[i]
    
        actions = (
            spadlstore[f"actions/game_{game_id}"]
            .merge(spadl.actiontypes_df(), how="left")
            .merge(spadl.results_df(), how="left")
            .merge(spadl.bodyparts_df(), how="left")
            .merge(spadlstore["players"], how="left")
            .merge(spadlstore["teams"], how="left")
        )
        
        actions["player_name"] = actions[["nickname", "player_name"]].apply(lambda x: x[0] if x[0] else x[1], axis=1)
        del actions['nickname']
        
        VAEP_df = pd.concat([VAEP_df,actions],axis=0)

In [298]:
VAEP_df = VAEP_df.reset_index().drop(columns=['index'])

In [299]:
VAEP_df

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name,player_name,team_name
0,3857256,4acb4fd2-f46a-4d73-993c-e06597873924,1,0.0,773,5545.0,52.058824,33.655696,67.852941,36.754430,0,1,5,0,pass,success,foot_right,Breel Embolo,Switzerland
1,3857256,0e5c867f-8c40-4754-bc81-014c8cf88b8d,1,1.0,773,6983.0,67.852941,36.754430,70.852941,38.303797,21,1,0,1,dribble,success,foot,Remo Freuler,Switzerland
2,3857256,f21fda3d-cb2c-4608-af2d-201d1ad8101c,1,3.0,773,6983.0,70.852941,38.303797,68.558824,61.286076,0,1,5,2,pass,success,foot_right,Remo Freuler,Switzerland
3,3857256,a54c6781-c843-40bb-9973-cdef963cb218,1,5.0,773,7796.0,68.558824,61.286076,68.558824,61.286076,21,1,0,3,dribble,success,foot,Silvan Widmer,Switzerland
4,3857256,57654ce3-e311-49c4-9ba7-4fab2ec66f71,1,6.0,773,7796.0,68.558824,61.286076,85.500000,52.850633,0,1,5,4,pass,success,foot_right,Silvan Widmer,Switzerland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313176,3835319,NaN,2,2812.0,865,18999.0,61.941176,58.617722,36.264706,56.637975,21,1,0,2045,dribble,success,foot,Leah Williamson,England Women's
313177,3835319,d3ff5dbc-5d29-43b9-8e0a-a2c01b513c47,2,2814.0,865,18999.0,36.264706,56.637975,23.382353,40.025316,0,1,5,2046,pass,success,foot_right,Leah Williamson,England Women's
313178,3835319,29031380-9530-4244-83b4-d8eedef99898,2,2816.0,865,31538.0,23.382353,40.025316,23.205882,42.521519,21,1,0,2047,dribble,success,foot,Mary Alexandra Earps,England Women's
313179,3835319,4a66ea1a-c99d-4887-9cf2-f0e9a1237061,2,2817.0,865,31538.0,23.205882,42.521519,47.029412,8.263291,0,1,5,2048,pass,success,foot_right,Mary Alexandra Earps,England Women's


### ii) Events

- VAEP_df에는 패스의 끝 위치가 없으므로 해당 데이터에서 불러오기
- StatsBomb360 Frame데이터의 위치정보가 VAEP_df위치정보랑 매치가 되지 않음(기준이나 거리 단위가 다름)

In [300]:
Events_df  = pd.DataFrame()
for i in tqdm.tqdm(MATCH_ID_list):
    Events_df = pd.concat([Events_df,sb.events(match_id=i)],axis=0)
Events_df= Events_df.reset_index().drop(columns=['level_0'])

  0%|          | 0/146 [00:00<?, ?it/s]

100%|██████████| 146/146 [00:53<00:00,  2.72it/s]


In [301]:
Events_df

,50_50,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,goalkeeper_shot_saved_to_post,pass_miscommunication,shot_saved_off_target,shot_saved_to_post,dribble_no_touch,shot_redirect,foul_won_penalty,shot_follows_dribble,player_off_permanent,goalkeeper_lost_in_play
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
532475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
532476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
532477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### iii) StatsBomb360 Frames data

- 플레이를 수행한 선수 뿐 아니라 주위의 선수들도 파악하기 위한 데이터

In [302]:
frame_360_df  = pd.DataFrame()
for i in tqdm.tqdm(MATCH_ID_list):
    frame_360_df = pd.concat([frame_360_df,pd.read_json(f"three-sixty/{i}.json")],axis=0)
frame_360_df = frame_360_df.reset_index().drop(columns=['index'])

100%|██████████| 146/146 [00:25<00:00,  5.81it/s]


In [303]:
frame_360_df

,event_uuid,visible_area,freeze_frame
0,5447ccd1-d408-44cf-8db4-31cbf6b612c1,"[1.4925415889868199, 80.0, 48.3931381525754, 0...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,366e4a09-2520-4590-9b0a-831c6c93b9e0,"[80.4178188804624, 74.8214573468144, 59.261574...","[{'teammate': True, 'actor': False, 'keeper': ..."
2,f39b7fcb-405f-4122-81cc-5db3a4483d39,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
3,9684f115-1ad7-4830-8e2f-acfb500464e5,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
4,dd14ae13-c274-421e-a4a1-a93f0e446980,"[58.3581817549259, 72.7511216981991, 47.540488...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...
462619,0b6ec289-35ef-404c-a0dd-025c34e14f08,"[73.7029019359023, 80.0, 68.5206661205348, 29....","[{'teammate': False, 'actor': False, 'keeper':..."
462620,1fe2951e-13b6-4aa3-982f-b69719b11269,"[0.0, 80.0, 0.0, 47.4838734697957, 23.73080008...","[{'teammate': True, 'actor': False, 'keeper': ..."
462621,5b7f534d-d0ff-44de-9fc3-5cabdd81716b,"[102.969547428233, 64.4983052198927, 77.370270...","[{'teammate': False, 'actor': False, 'keeper':..."
462622,ed487f52-f330-4dad-9b49-f36c5ff59c80,"[99.1695520058694, 69.9983052198927, 73.570275...","[{'teammate': False, 'actor': False, 'keeper':..."


# 3. 세 종류의 데이터 병합

- 360데이터를 기준으로 동일한 event_id끼리 병합한다.

### i) VAEP데이터과 360데이터를 병합하기

In [304]:
VAEP_360_df = pd.merge(left=VAEP_df,right=frame_360_df,left_on='original_event_id',right_on='event_uuid',how='right')
VAEP_360_df

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,...,bodypart_id,action_id,type_name,result_name,bodypart_name,player_name,team_name,event_uuid,visible_area,freeze_frame
0,3835340.0,5447ccd1-d408-44cf-8db4-31cbf6b612c1,1.0,1.0,864.0,10167.0,67.235294,44.070886,67.764706,44.070886,...,1.0,1.0,pass,fail,head,Tatiana Pinto,Portugal Women's,5447ccd1-d408-44cf-8db4-31cbf6b612c1,"[1.4925415889868199, 80.0, 48.3931381525754, 0...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,3835340.0,366e4a09-2520-4590-9b0a-831c6c93b9e0,1.0,3.0,858.0,26093.0,66.970588,44.845570,62.470588,42.435443,...,5.0,2.0,pass,fail,foot_right,Gun Nathalie Björn,Sweden Women's,366e4a09-2520-4590-9b0a-831c6c93b9e0,"[80.4178188804624, 74.8214573468144, 59.261574...","[{'teammate': True, 'actor': False, 'keeper': ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f39b7fcb-405f-4122-81cc-5db3a4483d39,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
3,3835340.0,9684f115-1ad7-4830-8e2f-acfb500464e5,1.0,4.0,864.0,32054.0,63.264706,41.660759,64.323529,51.645570,...,0.0,3.0,dribble,success,foot,Jéssica da Silva,Portugal Women's,9684f115-1ad7-4830-8e2f-acfb500464e5,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dd14ae13-c274-421e-a4a1-a93f0e446980,"[58.3581817549259, 72.7511216981991, 47.540488...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0b6ec289-35ef-404c-a0dd-025c34e14f08,"[73.7029019359023, 80.0, 68.5206661205348, 29....","[{'teammate': False, 'actor': False, 'keeper':..."
462620,3788758.0,1fe2951e-13b6-4aa3-982f-b69719b11269,2.0,2960.0,911.0,18617.0,19.323529,33.569620,25.764706,38.820253,...,1.0,1958.0,clearance,success,head,Mykola Matvienko,Ukraine,1fe2951e-13b6-4aa3-982f-b69719b11269,"[0.0, 80.0, 0.0, 47.4838734697957, 23.73080008...","[{'teammate': True, 'actor': False, 'keeper': ..."
462621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5b7f534d-d0ff-44de-9fc3-5cabdd81716b,"[102.969547428233, 64.4983052198927, 77.370270...","[{'teammate': False, 'actor': False, 'keeper':..."
462622,3788758.0,ed487f52-f330-4dad-9b49-f36c5ff59c80,2.0,2961.0,2358.0,30512.0,25.764706,38.820253,0.000000,42.177215,...,5.0,1959.0,shot,fail,foot_right,Aleksandar Trajkovski,North Macedonia,ed487f52-f330-4dad-9b49-f36c5ff59c80,"[99.1695520058694, 69.9983052198927, 73.570275...","[{'teammate': False, 'actor': False, 'keeper':..."


In [305]:
VAEP_360_df[VAEP_360_df['type_name'] == 'pass'][['type_name','result_name']]

,type_name,result_name
0,pass,fail
1,pass,fail
10,pass,success
13,pass,fail
26,pass,success
...,...,...
462602,pass,success
462604,pass,success
462606,pass,fail
462609,pass,success


- 360데이터의 모든 pass데이터의 결과유무가 들어있음(Nan값 없음)

In [306]:
print(VAEP_360_df[VAEP_360_df['type_name'] == 'pass']['result_name'].isna().sum())
print(VAEP_360_df[VAEP_360_df['type_name'] == 'pass']['result_name'].notna().sum())

0
114896


In [307]:
VAEP_360_df[VAEP_360_df['type_name'] == 'pass']['result_name'].value_counts()

success    96275
fail       18246
offside      375
Name: result_name, dtype: int64

### ii) Events과 360데이터를 병합하기

In [308]:
event_360_df = pd.merge(left=Events_df,right=frame_360_df,left_on='id',right_on='event_uuid',how='right')

In [309]:
event_360_df

,50_50,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,shot_saved_to_post,dribble_no_touch,shot_redirect,foul_won_penalty,shot_follows_dribble,player_off_permanent,goalkeeper_lost_in_play,event_uuid,visible_area,freeze_frame
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5447ccd1-d408-44cf-8db4-31cbf6b612c1,"[1.4925415889868199, 80.0, 48.3931381525754, 0...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,366e4a09-2520-4590-9b0a-831c6c93b9e0,"[80.4178188804624, 74.8214573468144, 59.261574...","[{'teammate': True, 'actor': False, 'keeper': ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f39b7fcb-405f-4122-81cc-5db3a4483d39,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
3,NaN,NaN,NaN,NaN,"[47.1, 61.0]",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9684f115-1ad7-4830-8e2f-acfb500464e5,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dd14ae13-c274-421e-a4a1-a93f0e446980,"[58.3581817549259, 72.7511216981991, 47.540488...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462619,NaN,Incomplete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0b6ec289-35ef-404c-a0dd-025c34e14f08,"[73.7029019359023, 80.0, 68.5206661205348, 29....","[{'teammate': False, 'actor': False, 'keeper':..."
462620,NaN,NaN,NaN,NaN,NaN,NaN,Head,True,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1fe2951e-13b6-4aa3-982f-b69719b11269,"[0.0, 80.0, 0.0, 47.4838734697957, 23.73080008...","[{'teammate': True, 'actor': False, 'keeper': ..."
462621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5b7f534d-d0ff-44de-9fc3-5cabdd81716b,"[102.969547428233, 64.4983052198927, 77.370270...","[{'teammate': False, 'actor': False, 'keeper':..."
462622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ed487f52-f330-4dad-9b49-f36c5ff59c80,"[99.1695520058694, 69.9983052198927, 73.570275...","[{'teammate': False, 'actor': False, 'keeper':..."


- VAEP_360_df[VAEP_360_df['type_name'] == 'pass'].shape = (114896, 22) : 총 114,896개의 pass데이터가 나온다.
- 그러나, event_360_df는 123,510 pass데이터가 나옴 
- 이유는 패스의 정의가 약간 다르기 때문 : VAEP_360_df는 골킥, 프리킥은 pass로 취급안하지만, event_360_df는 골킥, 프리킥을 Type에서는 같은 Pass로 취급하고 sub_pass_type에서 따로 분리함

In [310]:
event_360_df[event_360_df['type'] == 'Pass'].shape

(123510, 116)

In [311]:
event_360_df[event_360_df['type'] == 'Pass'][['type','pass_end_location']]

,type,pass_end_location
0,Pass,"[43.2, 52.2]"
1,Pass,"[71.8, 30.7]"
7,Pass,"[32.7, 53.9]"
10,Pass,"[20.1, 39.1]"
13,Pass,"[35.1, 13.0]"
...,...,...
462604,Pass,"[54.8, 44.3]"
462606,Pass,"[43.4, 41.3]"
462609,Pass,"[84.7, 54.7]"
462618,Pass,"[96.4, 64.1]"


- 360데이터의 모든 pass데이터의 끝 좌표가 들어있음(Nan값 없음)

In [312]:
print(event_360_df[event_360_df['type'] == 'Pass']['pass_end_location'].isna().sum())
print(event_360_df[event_360_df['type'] == 'Pass']['pass_end_location'].notna().sum())

0
123510


### iii) VAEP_360_df과 event_360_df데이터 결합하기

- 결합함으로써 모든 pass의 결과유무 & 끝 좌표를 얻을 수 있음

In [313]:
All_df = pd.merge(left=VAEP_360_df,right=event_360_df,left_on='event_uuid',right_on='event_uuid',how='right')

In [314]:
All_df

,game_id,original_event_id,period_id,time_seconds,team_id_x,player_id_x,start_x,start_y,end_x,end_y,...,shot_saved_off_target,shot_saved_to_post,dribble_no_touch,shot_redirect,foul_won_penalty,shot_follows_dribble,player_off_permanent,goalkeeper_lost_in_play,visible_area_y,freeze_frame_y
0,3835340.0,5447ccd1-d408-44cf-8db4-31cbf6b612c1,1.0,1.0,864.0,10167.0,67.235294,44.070886,67.764706,44.070886,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[1.4925415889868199, 80.0, 48.3931381525754, 0...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,3835340.0,366e4a09-2520-4590-9b0a-831c6c93b9e0,1.0,3.0,858.0,26093.0,66.970588,44.845570,62.470588,42.435443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[80.4178188804624, 74.8214573468144, 59.261574...","[{'teammate': True, 'actor': False, 'keeper': ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
3,3835340.0,9684f115-1ad7-4830-8e2f-acfb500464e5,1.0,4.0,864.0,32054.0,63.264706,41.660759,64.323529,51.645570,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[58.3581817549259, 72.7511216981991, 47.540488...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[73.7029019359023, 80.0, 68.5206661205348, 29....","[{'teammate': False, 'actor': False, 'keeper':..."
462620,3788758.0,1fe2951e-13b6-4aa3-982f-b69719b11269,2.0,2960.0,911.0,18617.0,19.323529,33.569620,25.764706,38.820253,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 80.0, 0.0, 47.4838734697957, 23.73080008...","[{'teammate': True, 'actor': False, 'keeper': ..."
462621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[102.969547428233, 64.4983052198927, 77.370270...","[{'teammate': False, 'actor': False, 'keeper':..."
462622,3788758.0,ed487f52-f330-4dad-9b49-f36c5ff59c80,2.0,2961.0,2358.0,30512.0,25.764706,38.820253,0.000000,42.177215,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[99.1695520058694, 69.9983052198927, 73.570275...","[{'teammate': False, 'actor': False, 'keeper':..."


# 4. All_df 데이터 탐색 및 이상여부 확인

- type_name : VAEP_df에서 action_type정의
- type : Events_df에서 주 action_type 정의
- pass_type : Event_df에서 서브 action_type정의
- result_name : VAEP_df에서 pass결과 유무 정의
- pass_outcome : Events_df에서 pass결과 유무 정의

In [315]:
All_df[['type_name','type','pass_type','result_name','pass_outcome']]

,type_name,type,pass_type,result_name,pass_outcome
0,pass,Pass,Recovery,fail,Incomplete
1,pass,Pass,Recovery,fail,Incomplete
2,NaN,Ball Recovery,NaN,NaN,NaN
3,dribble,Carry,NaN,success,NaN
4,NaN,Pressure,NaN,NaN,NaN
...,...,...,...,...,...
462619,NaN,Ball Receipt*,NaN,NaN,NaN
462620,clearance,Clearance,NaN,success,NaN
462621,NaN,Ball Recovery,NaN,NaN,NaN
462622,shot,Shot,NaN,fail,NaN


### i) pass타입 분석

- type_name : VAEP_df는 프리킥, 골킥은 따로 분류
- type, pass_type : Events_df는 큰 type은 같은 Pass로 분류하고, 서브 pass_type에서 따로 분류

In [316]:
All_df.loc[All_df['type']=='Pass',['type_name','type','pass_type']]

,type_name,type,pass_type
0,pass,Pass,Recovery
1,pass,Pass,Recovery
7,freekick_short,Pass,Free Kick
10,pass,Pass,NaN
13,pass,Pass,NaN
...,...,...,...
462604,pass,Pass,NaN
462606,pass,Pass,NaN
462609,pass,Pass,Recovery
462618,pass,Pass,NaN


### ii) pass결과유무 분석

- result_name : VAEP_df는 패스 결과 유무를 성공/실패/옵사이드로 분류
- pass_outcome : Events_df는 Nan은 성공, Nan이 아닌 것은 각각의 이유로 분류
1. Imcomplete : Ball does not reach a teammate and is still in play
2. Out : Ball goes out of bounds         
3. Unknown : Outcome is unknown (i.e. foul was called while in mid-flight)
4. Pass Offside : Ball reaches teammate but pass is judged offside
5. Injury Clearance : Ball is played out of bounds to stop play for an injury

In [317]:
All_df[All_df['type']=='Pass']['result_name'].value_counts()

success    101462
fail        21643
offside       405
Name: result_name, dtype: int64

In [318]:
All_df[All_df['type']=='Pass']['pass_outcome'].value_counts()

Incomplete          19553
Out                  2090
Unknown               600
Pass Offside          405
Injury Clearance       77
Name: pass_outcome, dtype: int64

In [319]:
All_df[All_df['type']=='Pass'][['result_name','pass_outcome']]

,result_name,pass_outcome
0,fail,Incomplete
1,fail,Incomplete
7,success,NaN
10,success,NaN
13,fail,Incomplete
...,...,...
462604,success,NaN
462606,fail,Incomplete
462609,success,NaN
462618,offside,Pass Offside


##### pass_outcome=Nan이면 result_name은 무엇일까?

- pass_outcome이 Nan인 값들은 실제로 모두 result_name=success인 것을 확인함

In [320]:
pass_df = All_df[All_df['type']=='Pass'].reset_index()
pass_df.shape

(123510, 138)

In [321]:
pass_df.loc[pass_df['pass_outcome'].notna(),'result_name'].value_counts()

fail       21643
success      677
offside      405
Name: result_name, dtype: int64

##### pass_outcome=Nan이 아니면 result_name은 무엇일까?

- pass_outcome이 Nan이 아닌 것은 무조건 result_name=fail인 것은 아님!
- Events_df의 pass_outcome에서 Unknown, Injury Clearance인 것도 실제로는 result_name = success 존재함

In [322]:
pass_df.loc[pass_df['pass_outcome'].notna(),'result_name'].value_counts()

fail       21643
success      677
offside      405
Name: result_name, dtype: int64

In [323]:
lst = []
for i in range(len(pass_df)):
    if pass_df.loc[i,'pass_outcome'] == pass_df.loc[i,'pass_outcome']:
        if (pass_df.loc[i,'result_name'] =='success'):
            lst.append(i)
len(lst)

677

In [324]:
pass_df[pass_df['pass_outcome'] == "Incomplete"]['result_name'].value_counts()

fail    19553
Name: result_name, dtype: int64

In [325]:
lst = []
for i in range(len(pass_df)):
    if pass_df.loc[i,'pass_outcome'] == "incomplete":
        if (pass_df.loc[i,'result_name'] =='success'):
            lst.append(i)
len(lst)

0

In [326]:
pass_df.loc[lst]['pass_outcome'].value_counts()

Series([], Name: pass_outcome, dtype: int64)

# 5. 최종 pass데이터 결론

- 총 123,510의 pass데이터가 존재함
- 패스 끝 위치와 결과유무는 알아서 labeling은 가능하지만, 모든 선수의 위치를 알 수 없어서 속도벡터를 구할 수 없을 것 같음

In [327]:
All_df.columns

Index(['game_id', 'original_event_id', 'period_id', 'time_seconds',
       'team_id_x', 'player_id_x', 'start_x', 'start_y', 'end_x', 'end_y',
       ...
       'shot_saved_off_target', 'shot_saved_to_post', 'dribble_no_touch',
       'shot_redirect', 'foul_won_penalty', 'shot_follows_dribble',
       'player_off_permanent', 'goalkeeper_lost_in_play', 'visible_area_y',
       'freeze_frame_y'],
      dtype='object', length=137)

In [328]:
col = ['location','type','pass_end_location','result_name','freeze_frame_x']
pass_df[col]

,location,type,pass_end_location,result_name,freeze_frame_x
0,"[43.8, 52.2]",Pass,"[43.2, 52.2]",fail,"[{'teammate': True, 'actor': False, 'keeper': ..."
1,"[76.9, 27.9]",Pass,"[71.8, 30.7]",fail,"[{'teammate': True, 'actor': False, 'keeper': ..."
2,"[46.4, 60.5]",Pass,"[32.7, 53.9]",success,"[{'teammate': True, 'actor': True, 'keeper': F..."
3,"[34.1, 51.2]",Pass,"[20.1, 39.1]",success,"[{'teammate': True, 'actor': True, 'keeper': F..."
4,"[22.9, 34.8]",Pass,"[35.1, 13.0]",fail,"[{'teammate': True, 'actor': True, 'keeper': T..."
...,...,...,...,...,...
123505,"[49.9, 57.9]",Pass,"[54.8, 44.3]",success,"[{'teammate': True, 'actor': False, 'keeper': ..."
123506,"[54.8, 44.3]",Pass,"[43.4, 41.3]",fail,"[{'teammate': True, 'actor': False, 'keeper': ..."
123507,"[76.7, 38.8]",Pass,"[84.7, 54.7]",success,"[{'teammate': False, 'actor': False, 'keeper':..."
123508,"[85.6, 70.6]",Pass,"[96.4, 64.1]",offside,"[{'teammate': True, 'actor': False, 'keeper': ..."


# 번외. 접근 방식을 아예 다르게 라벨링함

- 패스시 다음 소유팀이 변경된다면 패스실패로 라벨링
- 해당 방식으로 라벨링시 처음 방식과 다른 결과가 출력되는 경우가 종종 등장함
- 실제로는 기존 방식이 정답이라고 생각함(아래 이유도 설명)

In [329]:
other_df  = pd.DataFrame()
for i in tqdm.tqdm(MATCH_ID_list):
    other_df = pd.concat([other_df,sb.events(match_id=i)],axis=0)
other_df= other_df.reset_index().drop(columns=['level_0'])

100%|██████████| 146/146 [02:03<00:00,  1.18it/s]


In [330]:
other_df.pass_outcome.value_counts()

Incomplete          24185
Out                  2557
Unknown               791
Pass Offside          480
Injury Clearance      131
Name: pass_outcome, dtype: int64

In [331]:
for i in tqdm.tqdm(range(len(other_df))):
    if other_df.loc[i,'type'] == 'Pass':
        #Nan인 경우 레이블링
        if other_df.loc[i,'pass_outcome'] != other_df.loc[i,'pass_outcome']:
            #패스가 끊겨서 소유팀이 넘어감
            if other_df.loc[i,'possession_team'] != other_df.loc[i+1,'possession_team']:
                other_df.loc[i,'pass_result'] = False
            else:
                other_df.loc[i,'pass_result'] = True

  0%|          | 0/532479 [00:00<?, ?it/s]

100%|██████████| 532479/532479 [19:07<00:00, 464.03it/s]


In [332]:
other_df['pass_result'].value_counts()

True     118951
False      5491
Name: pass_result, dtype: int64

In [333]:
all_other_df = pd.merge(left=other_df,right=frame_360_df,left_on='id',right_on='event_uuid',how='right')

In [334]:
all_other_df

,50_50,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,dribble_no_touch,shot_redirect,foul_won_penalty,shot_follows_dribble,player_off_permanent,goalkeeper_lost_in_play,pass_result,event_uuid,visible_area,freeze_frame
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5447ccd1-d408-44cf-8db4-31cbf6b612c1,"[1.4925415889868199, 80.0, 48.3931381525754, 0...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,366e4a09-2520-4590-9b0a-831c6c93b9e0,"[80.4178188804624, 74.8214573468144, 59.261574...","[{'teammate': True, 'actor': False, 'keeper': ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f39b7fcb-405f-4122-81cc-5db3a4483d39,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
3,NaN,NaN,NaN,NaN,"[47.1, 61.0]",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9684f115-1ad7-4830-8e2f-acfb500464e5,"[0.0, 80.0, 0.0, 79.8958091901975, 45.85252978...","[{'teammate': True, 'actor': False, 'keeper': ..."
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dd14ae13-c274-421e-a4a1-a93f0e446980,"[58.3581817549259, 72.7511216981991, 47.540488...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462619,NaN,Incomplete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0b6ec289-35ef-404c-a0dd-025c34e14f08,"[73.7029019359023, 80.0, 68.5206661205348, 29....","[{'teammate': False, 'actor': False, 'keeper':..."
462620,NaN,NaN,NaN,NaN,NaN,NaN,Head,True,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1fe2951e-13b6-4aa3-982f-b69719b11269,"[0.0, 80.0, 0.0, 47.4838734697957, 23.73080008...","[{'teammate': True, 'actor': False, 'keeper': ..."
462621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5b7f534d-d0ff-44de-9fc3-5cabdd81716b,"[102.969547428233, 64.4983052198927, 77.370270...","[{'teammate': False, 'actor': False, 'keeper':..."
462622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ed487f52-f330-4dad-9b49-f36c5ff59c80,"[99.1695520058694, 69.9983052198927, 73.570275...","[{'teammate': False, 'actor': False, 'keeper':..."


In [350]:
o_df = all_other_df[all_other_df['type']=='Pass'][['pass_result','pass_outcome']]
o_df

,pass_result,pass_outcome
0,NaN,Incomplete
1,NaN,Incomplete
7,True,NaN
10,True,NaN
13,NaN,Incomplete
...,...,...
462604,True,NaN
462606,NaN,Incomplete
462609,False,NaN
462618,NaN,Pass Offside


In [351]:
v_df = All_df.loc[All_df['type']=='Pass',['result_name','pass_outcome']]
v_df

,result_name,pass_outcome
0,fail,Incomplete
1,fail,Incomplete
7,success,NaN
10,success,NaN
13,fail,Incomplete
...,...,...
462604,success,NaN
462606,fail,Incomplete
462609,success,NaN
462618,offside,Pass Offside


- 115,623번을 보면, 초기 제안한 result_name=success이지만, 여기서는 pass_result=False가 나옴
- 실제로 어땠는지 분석해볼 예정

In [353]:
n = 115630
pd.concat([o_df.loc[n-40:n],v_df.loc[n-40:n]],axis=1)

,pass_result,pass_outcome,result_name,pass_outcome
115592,True,NaN,success,NaN
115595,True,NaN,success,NaN
115598,True,NaN,success,NaN
115601,True,NaN,success,NaN
115605,NaN,Incomplete,fail,Incomplete
115607,True,NaN,success,NaN
115610,True,NaN,success,NaN
115614,NaN,Unknown,success,Unknown
115617,True,NaN,success,NaN
115620,True,NaN,success,NaN


In [365]:
#115,623의 Event_id알아내기
All_df.loc[115623].event_uuid, all_other_df.loc[115623].event_uuid

('9ed1439b-d3bd-40c8-aeec-7405c3c9ce83',
 '9ed1439b-d3bd-40c8-aeec-7405c3c9ce83')

In [374]:
#실제 Event데이터의 어디에 해당하는지 알아내기
Events_df[Events_df.id == '9ed1439b-d3bd-40c8-aeec-7405c3c9ce83']

,50_50,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,goalkeeper_shot_saved_to_post,pass_miscommunication,shot_saved_off_target,shot_saved_to_post,dribble_no_touch,shot_redirect,foul_won_penalty,shot_follows_dribble,player_off_permanent,goalkeeper_lost_in_play
133857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- 133,857경기를 보면 77->81로 넘어갈 때 소유팀이 바뀌어서 후자로 제안한 pass_result=False로 인식함
- 그러나 실제로 link에서 경기흐름(index기준)으로 보면 77->78은 뎀벨레선수가 패스를 받았다. 
- 경기가 index기준으로 정렬되어 있지 않다보니 이런 오류가 생기는데, 이것을 해결해줄 수도 있지만, <br/>기존  event_id별로 result_name를 추출하는 것이 더 편함<br/>
반박사례 사진 link : https://github.com/GunHeeJoe/EPV 


In [373]:
Events_df.loc[133857-3:133857+3][['index','possession_team','type','timestamp','period','player']]

,index,possession_team,type,timestamp,period,player
133854,68,France,Pass,00:01:34.817,1,Theo Bernard François Hernández
133855,71,France,Pass,00:01:55.993,1,Theo Bernard François Hernández
133856,74,France,Pass,00:02:02.182,1,Raphaël Varane
133857,77,France,Pass,00:02:04.612,1,Jules Koundé
133858,81,Argentina,Pass,00:02:11.124,1,Nicolás Alejandro Tagliafico
133859,84,Argentina,Pass,00:02:14.784,1,Alexis Mac Allister
133860,88,Argentina,Pass,00:02:16.985,1,Lionel Andrés Messi Cuccittini
